In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import sys
sys.path.append("../../utils/")

from utils import *

import time

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

ESTRATEGIA_DE_REBALANCEO = "NONE"
MODELO = "logreg"

NOMBRE_EXPERIMENTO = f"CIC17__split__v1__{ESTRATEGIA_DE_REBALANCEO}_pca4_{MODELO}__v1"
CARPETA_DATASET = "CIC17__split__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG LOGISTIC REGRESSION =====
LOGREG_C = 1.0
LOGREG_MAX_ITER = 1000
LOGREG_SOLVER = "lbfgs"
LOGREG_CLASS_WEIGHT = None
LOGREG_N_JOBS = -1

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 3

# ===== CONFIG REBALANCEO DENTRO DEL CV =====
TARGET_N = 10000
NEARMISS_VERSION = 1
SMOTE_K_NEIGHBORS = 5
ENN_N_NEIGHBORS = 3

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/CIC17__split__v1/CIC17__split__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/CIC17__split__v1/CIC17__split__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC17__split__v1__NONE_pca4_logreg__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(2016638, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,60146,974,6,454,258,0,75.666667,6,6,496919.917900,...,114,0,4,32,0.0,0.0,0,0,0.0,0
1,443,5227824,7,599,517,0,85.571429,152,0,143.654415,...,29200,237,3,32,0.0,0.0,0,0,0.0,0
2,53,49756,1,49,49,49,49.000000,77,77,2532.357907,...,-1,-1,0,32,0.0,0.0,0,0,0.0,0
3,80,170446,3,26,20,0,8.666667,4380,0,68215.153190,...,8192,229,2,20,0.0,0.0,0,0,0.0,2
4,53,170,2,58,29,29,29.000000,45,45,870588.235300,...,-1,-1,1,32,0.0,0.0,0,0,0.0,0


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,1676045
1,138277
2,102411
3,72555
4,8229
5,4745
6,4308
7,4182
8,2575


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (2016638, 47)
Shape y_train: (2016638,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("logreg", LogisticRegression(
        C=LOGREG_C,
        max_iter=LOGREG_MAX_ITER,
        solver=LOGREG_SOLVER,
        class_weight=LOGREG_CLASS_WEIGHT,
        n_jobs=LOGREG_N_JOBS,
        random_state=RANDOM_STATE
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",3
,"copy copy: bool, default=TrueIf False, data passed to fit are 

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
labels_globales = np.array(sorted(y_train.unique()))

resultados_folds = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):

    print("=" * 80)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 80)

    # =========================
    # Split del fold
    # =========================
    df_train_fold = df_train.iloc[train_idx].copy()
    df_val_fold = df_train.iloc[val_idx].copy()

    print("Shape train fold original:", df_train_fold.shape)
    print("Shape val fold original  :", df_val_fold.shape)
    print()

    # =========================
    # Rebalanceo SOLO sobre train fold
    # =========================
    df_train_fold_balanceado = rebalancear_train_fold(
        df_fold_train=df_train_fold,
        label_col=LABEL_COL,
        target_n=TARGET_N,
        random_state=RANDOM_STATE + fold,
        nearmiss_version=NEARMISS_VERSION,
        smote_k_neighbors=SMOTE_K_NEIGHBORS,
        estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
        enn_n_neighbors=ENN_N_NEIGHBORS
    )

    X_train_fold_bal = df_train_fold_balanceado.drop(columns=[LABEL_COL])
    y_train_fold_bal = df_train_fold_balanceado[LABEL_COL]

    X_val_fold = df_val_fold.drop(columns=[LABEL_COL])
    y_val_fold = df_val_fold[LABEL_COL]

    # =========================
    # Modelo nuevo para cada fold
    # =========================
    pipeline_fold = Pipeline([
        ("scaler", RobustScaler()),
        ("pca", PCA(n_components=N_COMPONENTS_PCA)),
        ("logreg", LogisticRegression(
            C=LOGREG_C,
            max_iter=LOGREG_MAX_ITER,
            solver=LOGREG_SOLVER,
            class_weight=LOGREG_CLASS_WEIGHT,
            n_jobs=LOGREG_N_JOBS,
            random_state=RANDOM_STATE + fold
        ))
    ])

    # =========================
    # Entrenamiento
    # =========================
    t0 = time.time()
    pipeline_fold.fit(X_train_fold_bal, y_train_fold_bal)
    fit_time = time.time() - t0

    # =========================
    # Validación
    # =========================
    t0 = time.time()
    y_pred_val = pipeline_fold.predict(X_val_fold)
    score_time = time.time() - t0

    roc_auc_val = calcular_roc_auc_multiclase_seguro(
        modelo=pipeline_fold,
        X_val=X_val_fold,
        y_val=y_val_fold,
        labels_globales=labels_globales
    )

    metricas_fold = {
        "fold": fold,

        "train_original_rows": int(df_train_fold.shape[0]),
        "train_balanceado_rows": int(df_train_fold_balanceado.shape[0]),
        "val_rows": int(df_val_fold.shape[0]),

        "accuracy": accuracy_score(y_val_fold, y_pred_val),

        "precision_weighted": precision_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),

        "precision_macro": precision_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),

        "mcc": matthews_corrcoef(y_val_fold, y_pred_val),
        "roc_auc": roc_auc_val,

        "fit_time": fit_time,
        "score_time": score_time
    }

    resultados_folds.append(metricas_fold)

    print("Métricas fold:")
    print(metricas_fold)
    print()

FOLD 1/5


Shape train fold original: (1613310, 48)
Shape val fold original  : (403328, 48)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     1340836
1      110622
2       81928
3       58044
4        6583
5        3796
6        3446
7        3346
8        2060
9        1246
10        941
11        418
12         23
13         14
14          7
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     1340836
1      110622
2       81928
3       58044
4        6583
5        3796
6        3446
7        3346
8        2060
9        1246
10        941
11        418
12         23
13         14
14          7
Name: count, dtype: int64
Shape final: (1613310, 48)



/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined


...

Shape train fold original: (1613311, 48)
Shape val fold original  : (403327, 48)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     1340836
1      110622
2       81929
3       58044
4        6583
5        3796
6        3446
7        3345
8        2060
9        1247
10        941
11        418
12         23
13         14
14          7
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     1340836
1      110622
2       81929
3       58044
4        6583
5        3796
6        3446
7        3345
8        2060
9        1247
10        941
11        418
12         23
13         14
14          7
Name: count, dtype: int64
Shape final: (1613311, 48)



/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined


Métricas fold:
{'fold': 5, 'train_original_rows': 1613311, 'train_balanceado_rows': 1613311, 'val_rows': 403327, 'accuracy': 0.8153706545805265, 'precision_weighted': 0.7137457049455985, 'recall_weighted': 0.8153706545805265, 'f1_weighted': 0.7550593715573946, 'precision_macro': 0.08606391347235752, 'recall_macro': 0.09307307427923016, 'f1_macro': 0.07647404931243199, 'mcc': 0.05886632780598715, 'roc_auc': nan, 'fit_time': 247.99616599082947, 'score_time': 0.07619380950927734}



In [11]:
df_folds = pd.DataFrame(resultados_folds)

df_folds

,fold,train_original_rows,train_balanceado_rows,val_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,1613310,1613310,403328,0.814900,0.713419,0.814900,0.754813,0.085262,0.092949,0.076137,0.057964,NaN,434.812013,0.121188
1,2,1613310,1613310,403328,0.814744,0.713743,0.814744,0.754782,0.085618,0.090932,0.075742,0.057098,NaN,388.320667,0.096573
2,3,1613310,1613310,403328,0.815302,0.714080,0.815302,0.755152,0.086410,0.093366,0.076748,0.059622,NaN,274.121968,0.077665
3,4,1613311,1613311,403327,0.814478,0.712851,0.814478,0.754304,0.084863,0.093377,0.075881,0.054731,NaN,251.750046,0.079190
4,5,1613311,1613311,403327,0.815371,0.713746,0.815371,0.755059,0.086064,0.093073,0.076474,0.058866,NaN,247.996166,0.076194


In [12]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "modelo": "LogisticRegression",
        "logreg_c": LOGREG_C,
        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_solver": LOGREG_SOLVER,
        "logreg_class_weight": LOGREG_CLASS_WEIGHT,
        "logreg_n_jobs": LOGREG_N_JOBS,
        "n_components_pca": N_COMPONENTS_PCA,
        "estrategia_rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "target_n": TARGET_N,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'CIC17__split__v1__NONE_pca4_logreg__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/CIC17__split__v1/CIC17__split__v1__train.csv',
 'shape_train': {'rows': 2016638, 'cols': 48},
 'parametros': {'modelo': 'LogisticRegression',
  'logreg_c': 1.0,
  'logreg_max_iter': 1000,
  'logreg_solver': 'lbfgs',
  'logreg_class_weight': None,
  'logreg_n_jobs': -1,
  'n_components_pca': 3,
  'estrategia_rebalanceo': 'NONE',
  'target_n': 10000,
  'nearmiss_version': 1,
  'smote_k_neighbors': 5,
  'enn_n_neighbors': 3},
 'metricas_media': {'accuracy': 0.8149588572321947,
  'precision_weighted': 0.7135679178671952,
  'recall_weighted': 0.8149588572321947,
  'f1_weighted': 0.7548220752728323,
  'precision_macro': 0.08564314053990692,
  'recall_macro': 0.09273942719981791,
  'f1_macro': 0.07619654007227164,
  'mcc': 0.057656214255999205,
  'roc_auc': nan,
  'fit_time': 319.40017194747924,
  'score_time': 0.09016203880310059},
 'metricas_std': {'a

In [13]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}\t"
    f"{summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}"
)

Accuracy	Precision weighted	Recall weighted	F1 weighted	Precision macro	Recall macro	F1 macro	MCC	ROC AUC
0.814959 ± 0.000377	0.713568 ± 0.000464	0.814959 ± 0.000377	0.754822 ± 0.000330	0.085643 ± 0.000616	0.092739 ± 0.001028	0.076197 ± 0.000416	0.057656 ± 0.001890	nan ± nan


In [14]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC17__split__v1__NONE_pca4_logreg__v1/CIC17__split__v1__NONE_pca4_logreg__v1__folds.csv


In [15]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC17__split__v1__NONE_pca4_logreg__v1/CIC17__split__v1__NONE_pca4_logreg__v1__summary_cv.json


In [16]:
df_folds

,fold,train_original_rows,train_balanceado_rows,val_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,1613310,1613310,403328,0.814900,0.713419,0.814900,0.754813,0.085262,0.092949,0.076137,0.057964,NaN,434.812013,0.121188
1,2,1613310,1613310,403328,0.814744,0.713743,0.814744,0.754782,0.085618,0.090932,0.075742,0.057098,NaN,388.320667,0.096573
2,3,1613310,1613310,403328,0.815302,0.714080,0.815302,0.755152,0.086410,0.093366,0.076748,0.059622,NaN,274.121968,0.077665
3,4,1613311,1613311,403327,0.814478,0.712851,0.814478,0.754304,0.084863,0.093377,0.075881,0.054731,NaN,251.750046,0.079190
4,5,1613311,1613311,403327,0.815371,0.713746,0.815371,0.755059,0.086064,0.093073,0.076474,0.058866,NaN,247.996166,0.076194


In [17]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(504160, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,443,93606787,18,3128,410,6,173.777778,1618,38,186.845426,...,256,8192,17,20,32068.8000,1749.725464,34107,30651,6.931172e+06,0
1,80,98331732,6,354,336,0,59.000000,4344,0,121.517233,...,0,235,3,20,21035.0000,0.000000,21035,21035,0.000000e+00,1
2,80,117175775,231,1427,401,0,6.177489,4584,0,5590.916723,...,29200,1039,4,32,206922.5455,636207.069100,2125159,14988,8.932177e+04,0
3,80,1285910,3,566,560,0,188.666667,1894,2,1923.929357,...,29200,15680,2,20,0.0000,0.000000,0,0,0.000000e+00,0
4,53,47941,1,44,44,44,44.000000,224,224,5590.204627,...,-1,-1,0,40,0.0000,0.000000,0,0,0.000000e+00,0


In [18]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,419012
1,34569
2,25603
3,18139
4,2057
5,1186
6,1077
7,1046
8,644


In [19]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (504160, 47)
Shape y_test: (504160,)


In [20]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [21]:
print("Rebalanceando todo el train original para entrenar el modelo final...")

df_train_balanceado_final = rebalancear_train_fold(
    df_fold_train=df_train,
    label_col=LABEL_COL,
    target_n=TARGET_N,
    random_state=RANDOM_STATE,
    nearmiss_version=NEARMISS_VERSION,
    smote_k_neighbors=SMOTE_K_NEIGHBORS,
    estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
    enn_n_neighbors=ENN_N_NEIGHBORS
)

X_train_balanceado_final = df_train_balanceado_final.drop(columns=[LABEL_COL])
y_train_balanceado_final = df_train_balanceado_final[LABEL_COL]

pipeline.fit(X_train_balanceado_final, y_train_balanceado_final)

print("Modelo final entrenado con todo el train rebalanceado.")
print("Train original   :", df_train.shape)
print("Train balanceado :", df_train_balanceado_final.shape)

Rebalanceando todo el train original para entrenar el modelo final...


Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     1676045
1      138277
2      102411
3       72555
4        8229
5        4745
6        4308
7        4182
8        2575
9        1558
10       1176
11        522
12         29
13         17
14          9
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     1676045
1      138277
2      102411
3       72555
4        8229
5        4745
6        4308
7        4182
8        2575
9        1558
10       1176
11        522
12         29
13         17
14          9
Name: count, dtype: int64
Shape final: (2016638, 48)



/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Modelo final entrenado con todo el train rebalanceado.
Train original   : (2016638, 48)
Train balanceado : (2016638, 48)


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted"
)

Predicciones en test generadas.
Número de predicciones: 504160


In [23]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.8149218502062837,
 'precision_weighted': 0.7135009072932693,
 'recall_weighted': 0.8149218502062837,
 'f1_weighted': 0.7547513330287648,
 'precision_macro': 0.08589064399411256,
 'recall_macro': 0.09369847846695165,
 'f1_macro': 0.07652146172880649,
 'roc_auc': 0.36234934255263107,
 'mcc': 0.05738322827190103}

In [24]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{metricas_test['accuracy']:.6f}\t"
    f"{metricas_test['precision_weighted']:.6f}\t"
    f"{metricas_test['recall_weighted']:.6f}\t"
    f"{metricas_test['f1_weighted']:.6f}\t"
    f"{metricas_test['precision_macro']:.6f}\t"
    f"{metricas_test['recall_macro']:.6f}\t"
    f"{metricas_test['f1_macro']:.6f}\t"
    f"{metricas_test['mcc']:.6f}\t"
    f"{metricas_test['roc_auc']:.6f}"
)

Accuracy	Precision weighted	Recall weighted	F1 weighted	Precision macro	Recall macro	F1 macro	MCC	ROC AUC
0.814922	0.713501	0.814922	0.754751	0.085891	0.093698	0.076521	0.057383	0.362349


In [25]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,408092,1516,2587,7,0,0,6180,630,0,0,0,0,0,0,0
1,33916,9,630,0,0,0,14,0,0,0,0,0,0,0,0
2,23217,0,2386,0,0,0,0,0,0,0,0,0,0,0,0
3,18139,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2043,0,2,0,0,0,6,6,0,0,0,0,0,0,0
5,1186,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,707,0,9,0,0,0,360,1,0,0,0,0,0,0,0
7,454,0,540,0,0,0,48,4,0,0,0,0,0,0,0
8,644,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,390,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [26]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========


              precision    recall  f1-score   support

           0       0.83      0.97      0.90    419012
           1       0.01      0.00      0.00     34569
           2       0.39      0.09      0.15     25603
           3       0.00      0.00      0.00     18139
           4       0.00      0.00      0.00      2057
           5       0.00      0.00      0.00      1186
           6       0.05      0.33      0.09      1077
           7       0.01      0.00      0.00      1046
           8       0.00      0.00      0.00       644
           9       0.00      0.00      0.00       390
          10       0.00      0.00      0.00       294
          11       0.00      0.00      0.00       130
          12       0.00      0.00      0.00         7
          13       0.00      0.00      0.00         4
          14       0.00      0.00      0.00         2

    accuracy                           0.81    504160
   macro avg       0.09      0.09      0.08    504160
weighted avg       0.71   

In [27]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "modelo": "LogisticRegression",
        "logreg_c": LOGREG_C,
        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_solver": LOGREG_SOLVER,
        "logreg_class_weight": LOGREG_CLASS_WEIGHT,
        "logreg_n_jobs": LOGREG_N_JOBS,
        "n_components_pca": N_COMPONENTS_PCA,
        "estrategia_rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "target_n": TARGET_N,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'CIC17__split__v1__NONE_pca4_logreg__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/CIC17__split__v1/CIC17__split__v1__test.csv',
 'shape_test': {'rows': 504160, 'cols': 48},
 'parametros': {'modelo': 'LogisticRegression',
  'logreg_c': 1.0,
  'logreg_max_iter': 1000,
  'logreg_solver': 'lbfgs',
  'logreg_class_weight': None,
  'logreg_n_jobs': -1,
  'n_components_pca': 3,
  'estrategia_rebalanceo': 'NONE',
  'target_n': 10000,
  'nearmiss_version': 1,
  'smote_k_neighbors': 5,
  'enn_n_neighbors': 3},
 'metricas_test': {'accuracy': 0.8149218502062837,
  'precision_weighted': 0.7135009072932693,
  'recall_weighted': 0.8149218502062837,
  'f1_weighted': 0.7547513330287648,
  'precision_macro': 0.08589064399411256,
  'recall_macro': 0.09369847846695165,
  'f1_macro': 0.07652146172880649,
  'mcc': 0.05738322827190103}}

In [28]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC17__split__v1__NONE_pca4_logreg__v1/CIC17__split__v1__NONE_pca4_logreg__v1__metricas_test.csv


In [29]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC17__split__v1__NONE_pca4_logreg__v1/CIC17__split__v1__NONE_pca4_logreg__v1__confusion_matrix_test.csv


In [30]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC17__split__v1__NONE_pca4_logreg__v1/CIC17__split__v1__NONE_pca4_logreg__v1__summary_test.json


In [31]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.8149588572321947, 'precision_weighted': 0.7135679178671952, 'recall_weighted': 0.8149588572321947, 'f1_weighted': 0.7548220752728323, 'precision_macro': 0.08564314053990692, 'recall_macro': 0.09273942719981791, 'f1_macro': 0.07619654007227164, 'mcc': 0.057656214255999205, 'roc_auc': nan, 'fit_time': 319.40017194747924, 'score_time': 0.09016203880310059}

TEST:
{'accuracy': 0.8149218502062837, 'precision_weighted': 0.7135009072932693, 'recall_weighted': 0.8149218502062837, 'f1_weighted': 0.7547513330287648, 'precision_macro': 0.08589064399411256, 'recall_macro': 0.09369847846695165, 'f1_macro': 0.07652146172880649, 'mcc': 0.05738322827190103}
